### `Importing Necessary`

In [1]:
import os
import json
import zipfile
import geopandas as gpd
from pathlib import Path
from bs4 import BeautifulSoup

### `Initializing Our Data Paths`

In [2]:
base = Path("../data/AGB")
kmz_path = base / "comp" / "forest_inventory_tapajos.kmz"
kml_path = base / "comp" / "doc.kml"  # temporary extraction
geojson_path = base / "comp" / "forest_inventory_tapajos.geojson"

### `Converting a KMZ (compressed KML) to GeoJSON`

In [3]:
# Extract KML from KMZ
with zipfile.ZipFile(kmz_path, "r") as kmz:
    kmz.extractall(os.path.dirname(kmz_path))

# Read KML using geopandas (requires fiona with KML support)
gdf = gpd.read_file(kml_path, driver="KML")
# Save as GeoJSON
gdf.to_file(geojson_path, driver="GeoJSON")

In [4]:
output_folder = "../data/AOIs"
os.makedirs(output_folder, exist_ok=True)

### `Create geojson files for each plots (30)`

In [5]:
# Load your GeoJSON
with open(geojson_path) as f:
    data = json.load(f)

# Loop through features and export individually
for i, feature in enumerate(data["features"], start=1):
    html_desc = feature["properties"]["Description"]
    soup = BeautifulSoup(html_desc, "html.parser")

    rows = soup.find_all("tr")

    # Extract key-value pairs
    props = {}
    for row in rows:
        cols = row.find_all("td")
        if len(cols) == 2:
            key = cols[0].text.strip()
            value = cols[1].text.strip()
            try:
                value = float(value)
            except:  # noqa: E722
                pass
            props[key] = value

    # Replace original properties
    feature["properties"] = props

    # Create a single-feature geojson
    single_geojson = {"type": "FeatureCollection", "features": [feature]}

    # Output path
    out_path = os.path.join(output_folder, f"aoi_plot_{i}.geojson")

    # Save file
    with open(out_path, "w") as f:
        json.dump(single_geojson, f, indent=2)

print("\n🎉 All AOIs of each plot has been created and saved!")


🎉 All AOIs of each plot has been created and saved!
